# V116 - Temporal audit

Pre-registered plan: every validation so far shuffles users across folds, but user IDs are sequential and chats carry timestamps, so test users may come from a later period than train users. This notebook audits time first: timestamp ranges per split, label drift by month, template emergence over time. Then it scores the word-only baseline on a temporal holdout (early users train, late users validate) beside the grouped number. A gap means distribution shift that template grouping cannot see. Audit only, no adoption.

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold

SEED = 4545
N_SPLITS = 5
rng = np.random.RandomState(SEED)

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v116")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve())

## Time audit, then temporal holdout

First-chat month per user orders the split. Chatless users join training unconditionally. Label means per month expose drift.

In [ ]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv", parse_dates=["timestamp"])
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)
first = chats.sort_values("timestamp").groupby("user_id").first()
fm = {u: pd.Timestamp(t) for u, t in zip(first.index, first["timestamp"])}
tr_months = pd.Series([fm.get(u) for u in train["user_id"]])
te_months = pd.Series([fm.get(u) for u in test_ids])
print("train chat months:", tr_months.dt.strftime("%Y-%m").value_counts().sort_index().to_dict())
print("test chat months:", te_months.dt.strftime("%Y-%m").value_counts().sort_index().to_dict())
print("train chatless:", tr_months.isna().sum(), "test chatless:", te_months.isna().sum())
ym = pd.DataFrame({"m": tr_months.dt.strftime("%Y-%m"), "top": Y.argmax(1)})
print("top-module mix by month (train):")
print(ym.groupby("m")["top"].agg(lambda s: s.value_counts().head(3).to_dict()).to_string())
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in list(train["user_id"])]
W = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit_transform(docs).toarray()

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

ts = tr_months.fillna(pd.Timestamp("2000-01-01"))
cut = ts.quantile(0.8)
late = np.where((tr_months > cut).fillna(False))[0]
early = np.setdiff1d(np.arange(n_train), late)
print(f"temporal split at {cut.date()}: early={len(early)} late={len(late)}")
oof_t = np.zeros_like(Y)
for j in range(len(MODS)):
    m = Ridge(alpha=1.0)
    m.fit(W[early], Y[early, j])
    oof_t[late, j] = m.predict(W[late])
print(f"temporal-holdout word-only: {ndcg5(Y[late], oof_t[late]).mean():.5f} (n={len(late)})")

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_user.get(u, [])) for u in train["user_id"]])
strata = Y.argmax(1)
oof_g = np.zeros_like(Y)
skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
for tr, va in skf.split(W, strata, fold_groups):
    for j in range(len(MODS)):
        m = Ridge(alpha=1.0)
        m.fit(W[tr], Y[tr, j])
        oof_g[va, j] = m.predict(W[va])
print(f"grouped-OOF word-only: {ndcg5(Y, oof_g).mean():.5f}")